---
## Step 4: USDA-Grounded Gram Weights

Replace heuristic unit conversions with evidence-based values.

**The problem:** `1 tbsp Beef` and `1 tbsp Mint Leaves` are **not** the same 15 g.  
- 1 tbsp fresh mint ≈ 2.5 g (light, airy herb)  
- 1 tbsp raw minced beef ≈ 15 g (dense muscle tissue)  
- 1 tbsp ground cumin ≈ 7 g (fine powder, partially air-filled)

**Three-tier grounding strategy:**

| Tier | Source | When used |
|---|---|---|
| **1 — USDA Foundation Foods** | `food_portion.csv` + `food.csv` | Ingredient matches a Foundation Food entry with the same unit |
| **2 — Food Science Reference** | USDA Handbook 8 / standard culinary references | Ingredient not in Foundation Foods but a known food (spices, fresh herbs, nuts, meats) |
| **3 — Generic density fallback** | Volume × density ≈ 1.0 g/ml | Liquids and items with no better reference |

**Multi-entry disambiguation:** When USDA has multiple entries for the same food (e.g. beef has 200 portion rows for steaks/roasts), we:
1. Filter to entries matching the **needed unit** (tablespoon/teaspoon/cup/medium/each)
2. Among those, prefer **plain raw whole-food** descriptions (no brand, no cooking modifier) 
3. Take the **median** gram-weight/amount to reduce outlier influence
4. Record which USDA description was selected

Ingredients where USDA Foundation Foods has the target unit → **Tier 1**  
Ingredients where USDA has no matching unit (e.g. spices, garlic, raw meats in tbsp) → **Tier 2**


In [4]:
import pandas as pd, numpy as np, re

# ── 4.1  Load SR Legacy (modifier column encodes the actual unit) ─────────
SR_DIR = 'FoodData_Central_sr_legacy_food_csv_2018-04'
sr_food = pd.read_csv(f'{SR_DIR}/food.csv')
sr_fp   = pd.read_csv(f'{SR_DIR}/food_portion.csv')
sr_fp   = sr_fp.merge(sr_food[['fdc_id', 'description']], on='fdc_id', how='left')

_UNIT_RE = [
    ('tsp',    r'\btsp\b|teaspoon'),
    ('tbsp',   r'\btbsp\b|tablespoon'),
    ('cup',    r'\bcup\b'),
    ('clove',  r'\bcloves?\b'),
    ('medium', r'\bmedium\b'),
    ('large',  r'\blarge\b'),
    ('small',  r'\bsmall\b'),
]

def _sr_unit(modifier) -> str:
    if pd.isna(modifier): return None
    m = modifier.lower()
    for unit, pat in _UNIT_RE:
        if re.search(pat, m): return unit
    return None

sr_fp['unit_type']  = sr_fp['modifier'].apply(_sr_unit)
sr_fp['desc_norm']  = sr_fp['description'].str.lower()
sr_fp['g_per_unit'] = sr_fp['gram_weight']          # amount is always 1.0 in SR Legacy
sr_fp = sr_fp[sr_fp['unit_type'].notna() & sr_fp['g_per_unit'].notna()]

# ── 4.2  Load Foundation Foods ────────────────────────────────────────────
FF_DIR = 'FoodData_Central_foundation_food_csv_2026-04-30'
ff_food = pd.read_csv(f'{FF_DIR}/food.csv')
ff_fp   = pd.read_csv(f'{FF_DIR}/food_portion.csv')
ff_mu   = pd.read_csv(f'{FF_DIR}/measure_unit.csv')
mu_map  = dict(zip(ff_mu['id'], ff_mu['name']))
ff_fp['unit_name'] = ff_fp['measure_unit_id'].map(mu_map)
ff_fp = ff_fp.merge(ff_food[['fdc_id', 'description']], on='fdc_id', how='left')
ff_fp['g_per_unit'] = ff_fp['gram_weight'] / ff_fp['amount'].replace(0, np.nan)

FF_UNIT_MAP = {
    'tablespoon': 'tbsp', 'Tablespoons': 'tbsp',
    'teaspoon': 'tsp',
    'cup': 'cup',
    'medium': 'medium', 'Onion': 'medium',
    'large': 'large', 'small': 'small',
    'egg': 'egg', 'each': 'medium',
    'fillet': 'fillet', 'filet': 'fillet',
}
ff_fp['unit_type'] = ff_fp['unit_name'].map(FF_UNIT_MAP)
ff_fp['desc_norm'] = ff_fp['description'].str.lower()
ff_fp = ff_fp[ff_fp['unit_type'].notna() & ff_fp['g_per_unit'].notna()]

print(f"SR Legacy:        {len(sr_fp):,} portion rows with parsed units")
print(f"Foundation Foods: {len(ff_fp):,} portion rows with mapped units")

# ── 4.3  Plainness scoring: prefer raw/simple over cooked/branded ─────────
def _plainness(desc: str) -> int:
    d = desc.lower()
    s = 0
    if re.search(r'\b(raw|fresh)\b', d):                                        s -= 3
    if re.search(r'\b(cooked|fried|roasted|baked|boiled|grilled|canned)\b', d): s += 3
    if re.search(r'\b(salted|seasoned|flavored|sweetened)\b', d):               s += 2
    if re.search(r'[A-Z]{3,}', desc[:40]):                                      s += 1  # ALL-CAPS brand
    return s

# ── 4.4  Synonym map: recipe ingredient → USDA search keyword ─────────────
# Only for cases where the ingredient name would not match USDA descriptions.
# Kept small (analogous to main.ipynb's manual_map). None = no USDA entry.
USDA_SYNONYMS = {
    'ghee'            : 'butter, salted',   # clarified butter; USDA has butter data
    'garam masala'    : 'curry powder',     # blend; "Spices, curry powder" is closest proxy
    'fresh coriander' : 'coriander leaf',   # "Spices, coriander leaf, dried"
    'mint leaves'     : 'spearmint',        # "Spearmint, dried"
    'lentils (dal)'   : 'lentils',          # strip parenthetical
    'mustard seeds'   : 'mustard seed',     # "Spices, mustard seed, ground"
    'sesame seeds'    : 'sesame seed',      # "Seeds, sesame seed kernels"
    'tamarind paste'  : 'tamarind',         # "Tamarinds, raw"
    'green chili'     : 'peppers, chili',
    'bell pepper'     : 'peppers, sweet',
    'paneer'          : 'cottage cheese',   # fresh cheese; no USDA paneer entry
    'fenugreek leaves': 'fenugreek',        # use seed data as dry-weight proxy
    'basmati rice'    : 'rice, white',
    'curry leaves'    : None,               # absent from USDA; fall through to generic
}

# ── 4.5  USDA lookup ──────────────────────────────────────────────────────
def _usda_g(keyword: str, unit: str, df: pd.DataFrame):
    """Keyword search → unit filter → median g/unit of 5 plainest matches."""
    mask = df['desc_norm'].str.contains(keyword.lower(), na=False, regex=False)
    sub  = df[mask & (df['unit_type'] == unit)].copy()
    if sub.empty: return None
    sub['_rank'] = sub['desc_norm'].apply(_plainness)
    vals = sub.nsmallest(5, '_rank')['g_per_unit'].dropna()
    return float(vals.median()) if not vals.empty else None

# Generic fallback — used only when USDA has no matching entry for the unit
_FALLBACK = {
    'tbsp': 15.0, 'tsp': 5.0, 'cup': 240.0, 'ml': 1.0,
    'g': 1.0, 'kg': 1000.0, 'clove': 4.0,
    'medium': 100.0, 'count_medium': 100.0,
    'large': 150.0, 'small': 60.0,
    'to_taste': 5.0, 'other': 100.0,
    'egg': 50.0, 'fillet': 150.0,
}

def get_grams(ingredient: str, unit: str) -> tuple:
    """Returns (g_per_unit, tier) where tier in {'sr_legacy','foundation_foods','generic'}."""
    ingr_l  = ingredient.lower().strip()
    ingr_cl = re.sub(r'\s*\(.*?\)', '', ingr_l).strip()   # strip parentheticals like "(dal)"
    lookup_unit = 'medium' if unit == 'count_medium' else unit

    if ingr_l in USDA_SYNONYMS:
        kw = USDA_SYNONYMS[ingr_l]
        keywords = [kw] if kw is not None else []
    else:
        keywords = list(dict.fromkeys(filter(None, [ingr_cl, ingr_l if ingr_l != ingr_cl else None])))

    for kw in keywords:
        g = _usda_g(kw, lookup_unit, sr_fp)
        if g is not None: return g, 'sr_legacy'
        g = _usda_g(kw, lookup_unit, ff_fp)
        if g is not None: return g, 'foundation_foods'

    return _FALLBACK.get(unit, _FALLBACK.get(lookup_unit, 100.0)), 'generic'

# ── 4.6  Parse quantities ─────────────────────────────────────────────────
def _parse_qty(qty_str: str) -> tuple:
    q = str(qty_str).lower().strip()
    if any(x in q for x in ['taste', 'needed', 'optional']): return 1.0, 'to_taste'
    m = re.match(r'^([\d.]+)', q)
    n = float(m.group(1)) if m else 1.0
    if   re.search(r'\bkg\b', q):                  return n, 'kg'
    elif re.search(r'\bg\b|grams?\b', q):           return n, 'g'
    elif re.search(r'\bml\b', q):                   return n, 'ml'
    elif re.search(r'tbsp|tablespoon', q):          return n, 'tbsp'
    elif re.search(r'tsp|teaspoon', q):             return n, 'tsp'
    elif re.search(r'\bcup\b', q):                  return n, 'cup'
    elif re.search(r'cloves?', q):                  return n, 'clove'
    elif re.search(r'\begg\b', q):                  return n, 'egg'
    elif re.search(r'fillet', q):                   return n, 'fillet'
    elif re.search(r'count_medium|medium', q):      return n, 'count_medium'
    elif re.search(r'\blarge\b', q):                return n, 'large'
    elif re.search(r'\bsmall\b', q):                return n, 'small'
    else:                                           return n, 'other'

ingr_df2 = ingr_df.copy()
ingr_df2['ingredient_name'] = ingr_df2['ingredient_name'].str.strip()

# Parse quantities if not already present
if 'qty_unit' not in ingr_df2.columns:
    qty_col = 'quantity' if 'quantity' in ingr_df2.columns else ingr_df2.columns[-1]
    parsed  = ingr_df2[qty_col].apply(
        lambda x: pd.Series(_parse_qty(x), index=['qty_amount', 'qty_unit'])
    )
    ingr_df2 = pd.concat([ingr_df2, parsed], axis=1)
elif 'qty_amount' not in ingr_df2.columns:
    ingr_df2['qty_amount'] = ingr_df2['quantity'].apply(
        lambda x: float(re.match(r'^([\d.]+)', str(x).strip()).group(1))
                  if re.match(r'^([\d.]+)', str(x).strip()) else 1.0
    )

# Apply USDA gram lookup (cached by unique ingredient × unit key)
_cache: dict = {}
def _cached(ingr: str, unit: str) -> tuple:
    k = (ingr.lower().strip(), unit)
    if k not in _cache:
        _cache[k] = get_grams(ingr, unit)
    return _cache[k]

res = ingr_df2.apply(
    lambda r: pd.Series(_cached(r['ingredient_name'], r['qty_unit']),
                        index=['g_per_unit', 'usda_tier']),
    axis=1
)
ingr_df2 = pd.concat([ingr_df2, res], axis=1)
ingr_df2['grams'] = ingr_df2['qty_amount'] * ingr_df2['g_per_unit']

# ── 4.7  Tier summary ─────────────────────────────────────────────────────
n_rows = len(ingr_df2)
print(f"\nProcessed {n_rows:,} rows  |  "
      f"{ingr_df2['recipe_id'].nunique()} recipes  |  "
      f"{ingr_df2['ingredient_name'].nunique()} unique ingredients")

print(f"\n=== USDA source tier ===")
tier_counts = ingr_df2['usda_tier'].value_counts()
for tier, cnt in tier_counts.items():
    print(f"  {tier:<20} {cnt:>7,}  ({cnt/n_rows*100:.1f}%)")

# ── 4.8  Grounded g/unit table ────────────────────────────────────────────
unit_table = (
    ingr_df2.groupby(['ingredient_name', 'qty_unit'])
    .agg(g_per_unit=('g_per_unit', 'first'),
         tier=('usda_tier', 'first'),
         n_rows=('grams', 'count'))
    .reset_index()
    .sort_values(['ingredient_name', 'qty_unit'])
)
print(f"\n=== Grounded g/unit table ({len(unit_table)} ingredient × unit combos) ===")
pd.set_option('display.max_colwidth', 55)
display(unit_table.reset_index(drop=True))

# ── 4.9  Recompute recipe-level CO₂e ─────────────────────────────────────
co2_lu  = match_df.set_index('ingredient')[['co2_kg_per_kg']]
ingr_df2 = ingr_df2.join(co2_lu, on='ingredient_name')
ingr_df2['co2e_g'] = ingr_df2.apply(
    lambda r: r['grams'] * (r['co2_kg_per_kg'] / 1000)
              if pd.notna(r['co2_kg_per_kg']) else 0.0, axis=1
)

recipe_co2 = (
    ingr_df2.groupby('recipe_id')
    .agg(
        co2e_kg=('co2e_g',         lambda x: x.sum() / 1000),
        n_ingr  =('ingredient_name','count'),
        n_sr    =('usda_tier',      lambda x: (x == 'sr_legacy').sum()),
        n_ff    =('usda_tier',      lambda x: (x == 'foundation_foods').sum()),
        n_gen   =('usda_tier',      lambda x: (x == 'generic').sum()),
    )
    .reset_index()
)
recipe_co2['usda_pct']    = ((recipe_co2['n_sr'] + recipe_co2['n_ff'])
                              / recipe_co2['n_ingr'] * 100).round(1)
recipe_co2['generic_pct'] = (recipe_co2['n_gen'] / recipe_co2['n_ingr'] * 100).round(1)

print(f"\n=== Recipe CO₂e (USDA-grounded) ===")
print(f"  Recipes analysed        : {len(recipe_co2)}")
print(f"  Median CO₂e / recipe    : {recipe_co2['co2e_kg'].median():.3f} kg CO₂eq")
print(f"  Avg USDA coverage       : {recipe_co2['usda_pct'].mean():.1f}%")
print(f"  Avg generic fallback    : {recipe_co2['generic_pct'].mean():.1f}%")

print(f"\nTop 10 highest-emission recipes:")
display(
    recipe_co2.nlargest(10, 'co2e_kg')
    [['recipe_id', 'co2e_kg', 'n_ingr', 'usda_pct', 'generic_pct']]
    .reset_index(drop=True)
)

# ── 4.10  Persist results ─────────────────────────────────────────────────
ingr_df2.to_csv('recipe_ingredients_grounded.csv', index=False)
recipe_co2.to_csv('recipe_co2.csv', index=False)
print(f"\nSaved: recipe_ingredients_grounded.csv  ({len(ingr_df2):,} rows)")
print(f"Saved: recipe_co2.csv  ({len(recipe_co2):,} rows)")


SR Legacy:        4,446 portion rows with parsed units
Foundation Foods: 7,093 portion rows with mapped units

Processed 103,908 rows  |  9997 recipes  |  58 unique ingredients

=== USDA source tier ===
  sr_legacy             54,981  (52.9%)
  generic               48,927  (47.1%)

=== Grounded g/unit table (71 ingredient × unit combos) ===


,ingredient_name,qty_unit,g_per_unit,tier,n_rows
0,All-Purpose Flour,g,1.0,generic,1608
1,Almonds,tbsp,9.1,sr_legacy,1116
2,Basmati Rice,g,1.0,generic,1549
3,Bay Leaf,tsp,0.6,sr_legacy,814
4,Beef,other,100.0,generic,892
...,...,...,...,...,...
66,Turmeric,tsp,3.0,sr_legacy,860
67,Vinegar,tbsp,14.9,sr_legacy,1350
68,Water,tbsp,15.0,sr_legacy,1353
69,Wheat Flour,g,1.0,generic,1588



=== Recipe CO₂e (USDA-grounded) ===
  Recipes analysed        : 9997
  Median CO₂e / recipe    : 0.246 kg CO₂eq
  Avg USDA coverage       : 52.6%
  Avg generic fallback    : 47.4%

Top 10 highest-emission recipes:


,recipe_id,co2e_kg,n_ingr,usda_pct,generic_pct
0,RCP09663,5.985002,18,55.6,44.4
1,RCP07413,5.980672,8,37.5,62.5
2,RCP05100,5.980455,9,66.7,33.3
3,RCP09571,5.979850,9,77.8,22.2
4,RCP07575,5.979533,8,62.5,37.5
5,RCP02996,5.975684,16,43.8,56.2
6,RCP08933,5.973319,7,71.4,28.6
7,RCP00893,5.968732,10,50.0,50.0
8,RCP00211,5.968170,5,40.0,60.0
9,RCP02867,5.956002,7,57.1,42.9



Saved: recipe_ingredients_grounded.csv  (103,908 rows)
Saved: recipe_co2.csv  (9,997 rows)
